A grade implementation
description:
Same multi-block builds and tiled structure as the B version. In addition, each warp reads 32 neighbouring cells of a row and packs them into one 32-bit word with __ballot_sync, so a single AND of two words tests 32 values of k at once. Each stage covers 128 values of k, which means 4× fewer block barriers.



Tests: 

n = 8; expected_degree = n >> 1;

n = 8, expected degree = 2:

n = 4096, expected degree = 82:
run 1:

run 2:

run 3:


In [ ]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

In [202]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b


In [203]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [204]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <cassert>  // for assert()
#include <cstddef>  // std::size_t type
#include <random>   // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph.
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && m < n * ( n - 1 ) );

    int const probability = ( 100 * m ) / ( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 100);

    edge_list_t random_edges;
    random_edges.reserve( 2 * m );

    for( node_t u = 0; u < n; ++u )
    {
        for( node_t v = u + 1; v < n; ++v )
        {
            auto const dice_roll = distrib( rng );
            if( dice_roll <= probability )
            {
                random_edges.push_back( make_int2( u, v ) );
                random_edges.push_back( make_int2( v, u ) );
            }
        }
    }

    random_edges.resize( random_edges.size() );


    return random_edges;
}

} // namespace a2
} // namespace csc485b


In [ ]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 */
struct DenseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

  /** Returns number of cells in the adjacency matrix. */
  __device__ __host__ __forceinline__
  std::size_t matrix_size() const { return n * n; }
};


namespace gpu {


/**
 * Constructs a DenseGraph from an input edge list of m edges.
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 */
__global__
void build_graph( DenseGraph g, edge_t const * edge_list, std::size_t m )
{
    // IMPLEMENT ME!

    // multiple blocks, each block handles a set of rows (a band)
    // no cell is ever tocuhed by more than one block
    
    // each block owns a contiguous band of rows. No cell is ever touched by two
    // blocks, so a block-level barrier (__syncthreads)
    std::size_t const rows_per_block = ( g.n + gridDim.x - 1 ) / gridDim.x;   // ceiling division
    std::size_t const row_begin      = blockIdx.x * rows_per_block;
    std::size_t const row_end        = ( row_begin + rows_per_block < g.n ) ? row_begin + rows_per_block : g.n;

    
    if( row_begin >= g.n ) return;

    // step 1: zero my rows. They are contiguous in memory, from cell row_begin*n to row_end*n.
    std::size_t const cell_begin = row_begin * g.n;
    std::size_t const cell_end   = row_end   * g.n;
    for( std::size_t i = cell_begin + threadIdx.x; i < cell_end; i += blockDim.x )
        g.adjacencyMatrix[ i ] = 0;

    // all of my rows are zero before any thread in my block writes a 1
    __syncthreads();

    // step 2: every block scans the whole edge list but only writes into its own rows
    for( std::size_t i = threadIdx.x; i < m; i += blockDim.x )
    {
        edge_t const e = edge_list[ i ];
        std::size_t const u = e.x, v = e.y;   // to size_t: avoids signed/unsigned comparisons

        if( u >= row_begin && u < row_end ) g.adjacencyMatrix[ u * g.n + v ] = 1;   // cell (u,v) is in row u
        if( v >= row_begin && v < row_end ) g.adjacencyMatrix[ v * g.n + u ] = 1;   // cell (v,u) is in row v
    }
    return;
}


__device__ unsigned int two_hop_blocks_done = 0;

/** Side length of the square output tile each block computes. */
constexpr int TILE       = 32;
/** Rows of threads (= warps) per block; each thread computes TILE / BLOCK_ROWS output cells. */
constexpr int BLOCK_ROWS = 8;
/** 32-bit words of k packed per row per stage: each stage covers 32 * K_WORDS values of k. */
constexpr int K_WORDS    = 4;





/**
  * Repopulates the adjacency matrix as a new graph that represents
  * the two-hop neighbourhood of input graph g
  *
  * Tiled + bit-packed: each block computes a TILE x TILE output tile. Input rows are packed
  * 32 cells per word with __ballot_sync into shared memory, so one AND tests 32 values of k.
  *
  */
__global__
void two_hop_reachability( DenseGraph g )
{
    // IMPLEMENT ME!
    constexpr int CELLS        = TILE / BLOCK_ROWS;            // 4 output cells per thread
    constexpr int WORDS_PER_SET = TILE * K_WORDS;              // words to pack for one side (i or j)

    __shared__ unsigned bits_i[ TILE ][ K_WORDS + 1 ];         // packed rows i0..i0+31; +1 pad: no bank conflicts
    __shared__ unsigned bits_j[ TILE ][ K_WORDS + 1 ];         // packed rows j0..j0+31 (row j == column j)

    std::size_t const n  = g.n;
    node_t *          A  = g.adjacencyMatrix;
    unsigned const    tx = threadIdx.x;                        // lane, 0..31
    unsigned const    ty = threadIdx.y;                        // warp index within the block
    std::size_t const i0 = std::size_t( blockIdx.y ) * TILE;
    std::size_t const j0 = std::size_t( blockIdx.x ) * TILE;

    int found[ CELLS ] = { 0 };

    // step 1: tiled, bit-packed squaring, reading bit 0 only
    for( std::size_t k0 = 0; k0 < n; k0 += 32 * K_WORDS ){
        // pack: 2 * WORDS_PER_SET words per stage, shared out across the warps.
        // The loop bounds depend only on ty, so every lane of a warp runs the same
        // iterations: the full-warp ballot is safe.
        for( int t = ty; t < 2 * WORDS_PER_SET; t += BLOCK_ROWS ){
            int const which = t / WORDS_PER_SET;               // 0 → i rows, 1 → j rows
            int const rem   = t % WORDS_PER_SET;
            int const row   = rem / K_WORDS;
            int const w     = rem % K_WORDS;

            std::size_t const grow = ( which == 0 ? i0 : j0 ) + row;
            std::size_t const k    = k0 + std::size_t( w ) * 32 + tx;
            int const bit = ( grow < n && k < n ) ? ( A[ grow * n + k ] & 1 ) : 0;   // coalesced 32-cell read

            unsigned const word = __ballot_sync( 0xffffffff, bit );                 // 32 cells → 1 word
            if( tx == 0 )
            {
                if( which == 0 ) bits_i[ row ][ w ] = word;
                else             bits_j[ row ][ w ] = word;
            }
        }
        __syncthreads();                                       // packed tiles ready

        // compute: one AND tests 32 values of k
        #pragma unroll
        for( int w = 0; w < K_WORDS; ++w ){
            unsigned const b = bits_j[ tx ][ w ];              // my column j = j0 + tx
            #pragma unroll
            for( int r = 0; r < CELLS; ++r )
                found[ r ] |= ( bits_i[ ty + r * BLOCK_ROWS ][ w ] & b ) != 0;   // same for whole warp → broadcast
        }

        // block vote: stop if every cell in the tile has a path
        int all = 1;
        #pragma unroll
        for( int r = 0; r < CELLS; ++r ) all &= found[ r ];
        if( __syncthreads_and( all ) ) break;
    }

    // write my cells: keep bit 0, resultds into bit 1
    std::size_t const j = j0 + tx;
    #pragma unroll
    for( int r = 0; r < CELLS; ++r ){
        std::size_t const i = i0 + ty + r * BLOCK_ROWS;
        if( i < n && j < n )
        {
            int const result = ( i != j ) ? found[ r ] : 0;   // remove the diagonal
            A[ i * n + j ] = ( A[ i * n + j ] & 1 ) | ( result << 1 );
        }
    }

    // grid-level sync: is last block
    __threadfence();
    __syncthreads();

    __shared__ bool is_last_block;
    if( tx == 0 && ty == 0 ){
        unsigned int const total_blocks = gridDim.x * gridDim.y;
        unsigned int const ticket       = atomicAdd( &two_hop_blocks_done, 1 );
        is_last_block = ( ticket == total_blocks - 1 );
    }
    __syncthreads();

    // step 2: last block shifts out the input bit
    if( is_last_block ){
        volatile node_t * V = A;
        unsigned const tid     = ty * blockDim.x + tx;
        unsigned const threads = blockDim.x * blockDim.y;

        for( std::size_t c = tid; c < g.matrix_size(); c += threads )
            V[ c ] = V[ c ] >> 1;

        if( tid == 0 ) two_hop_blocks_done = 0;
    }
   return;

}

} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */
struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n=|V| offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */
};


namespace gpu {

/**
 * Constructs a SparseGraph from an input edge list of m edges.
 *
 * @pre The pointers in SparseGraph g have already been allocated.
 */
__global__
void build_graph( SparseGraph g, edge_t const * edge_list, std::size_t m )
{
    // IMPLEMENT ME!
    // my band of rows row_begin, row_end
    std::size_t const rows_per_block = ( g.n + gridDim.x - 1 ) / gridDim.x;
    std::size_t const row_begin      = blockIdx.x * rows_per_block;
    std::size_t const row_end        = ( row_begin + rows_per_block < g.n ) ? row_begin + rows_per_block : g.n;

    // more blocks than rows, this whole block owns nothing and exits together
    if( row_begin >= g.n ) return;

    node_t *       start     = g.neighbours_start_at;
    unsigned const tid       = threadIdx.x;
    unsigned const lane      = tid & 31;
    unsigned const warp      = tid >> 5;
    unsigned const num_warps = blockDim.x >> 5;

    __shared__ int    edges_before;        // nunm edges whose source row is before my band
    __shared__ node_t warp_totals[ 32 ];   // scratch for the block-level scan

    // step 1: zero my rows degree counters
    for( std::size_t u = row_begin + tid; u < row_end; u += blockDim.x )
        start[ u ] = 0;
    if( tid == 0 ) edges_before = 0;
    __syncthreads();

    // step 2: one pass over all edges
    int before = 0;
    for( std::size_t i = tid; i < m; i += blockDim.x ){
        std::size_t const u = edge_list[ i ].x;
        if( u < row_begin )    ++before;                      // belongs to an earlier band
        else if( u < row_end ) atomicAdd( &start[ u ], 1 );   // my row count its degree
    }
    // warp-level reduction of before, then one shared atomic per warp
    for( int d = 16; d > 0; d >>= 1 )
        before += __shfl_down_sync( 0xffffffff, before, d );
    if( lane == 0 ) atomicAdd( &edges_before, before );
    __syncthreads();

    // step 3: inclusive scan of my bands degrees, offset by edges_before
    // processes the band in chunks of blockDim.x; carry = running total
    node_t carry = edges_before;
    for( std::size_t chunk = row_begin; chunk < row_end; chunk += blockDim.x ){
        std::size_t const u = chunk + tid;
        node_t x = ( u < row_end ) ? start[ u ] : 0;

        // warp level: inclusive scan with shuffles
        for( int d = 1; d < 32; d <<= 1 ) {
            node_t const y = __shfl_up_sync( 0xffffffff, x, d );
            if( lane >= d ) x += y;
        }
        if( lane == 31 ) warp_totals[ warp ] = x;
        __syncthreads();

        // block level: warp 0 scans the per-warp totals
        if( warp == 0 ) {
            node_t w = ( lane < num_warps ) ? warp_totals[ lane ] : 0;
            for( int d = 1; d < 32; d <<= 1 ) {
                node_t const y = __shfl_up_sync( 0xffffffff, w, d );
                if( lane >= d ) w += y;
            }
            warp_totals[ lane ] = w;
        }
        __syncthreads();

        x += carry + ( warp > 0 ? warp_totals[ warp - 1 ] : 0 );
        if( u < row_end ) start[ u ] = x;                     // = END of row u
        carry += warp_totals[ num_warps - 1 ];                // add this chunk's total
        __syncthreads();                                      // before warp_totals is reused
    }
    node_t const band_end = carry;                            // end of my last row

    // step 4: scatter. Decrementing turns start[u] from END into START of row u.
    for( std::size_t i = tid; i < m; i += blockDim.x ){
        edge_t const e = edge_list[ i ];
        std::size_t const u = e.x;
        if( u >= row_begin && u < row_end )
        {
            int const pos = atomicSub( &start[ u ], 1 ) - 1;
            g.neighbours[ pos ] = e.y;
        }
    }
    __syncthreads();

    // step 5: sort each of my rows insertion sort
    for( std::size_t u = row_begin + tid; u < row_end; u += blockDim.x ){
        node_t const begin = start[ u ];
        node_t const end   = ( u + 1 < row_end ) ? start[ u + 1 ] : band_end;
        for( node_t a = begin + 1; a < end; ++a )
        {
            node_t const key = g.neighbours[ a ];
            node_t b = a - 1;
            while( b >= begin && g.neighbours[ b ] > key ) { g.neighbours[ b + 1 ] = g.neighbours[ b ]; --b; }
            g.neighbours[ b + 1 ] = key;
        }
    }

    return;
}

/**
  * Repopulates the adjacency lists as a new graph that represents
  * the two-hop neighbourhood of input graph g
  */
__global__
void two_hop_reachability( SparseGraph g )
{
    // IMPLEMENT ME!
    // algorithm unknown
    return;
}

} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <chrono>   // for timing
#include <iostream> // std::cout, std::endl
#include <iterator> // std::ostream_iterator
#include <vector>

#include "dense_graph.h"
#include "sparse_graph.h"

#include "data_generator.h"
#include "data_types.h"

/**
 * Runs timing tests on a CUDA graph implementation.
 * Consists of independently constructing the graph and then
 * modifying it to its two-hop neighbourhood.
 */
template < typename DeviceGraph >
void run( DeviceGraph g, csc485b::a2::edge_t const * d_edges, std::size_t m )
{
    CUDA_CALL( cudaDeviceSynchronize() );   // also catches "no GPU" / earlier errors
    auto const build_start = std::chrono::high_resolution_clock::now();

    // multiple blocks: each block owns a band of rows
    csc485b::a2::gpu::build_graph<<< 4, 256 >>>( g, d_edges, m );
    CHECK_ERROR( "build_graph launch" );

    CUDA_CALL( cudaDeviceSynchronize() );
    auto const reachability_start = std::chrono::high_resolution_clock::now();

    dim3 const block( 32, 8 );                                    // TILE x BLOCK_ROWS
    dim3 const grid( ( g.n + 31 ) / 32, ( g.n + 31 ) / 32 );      // one block per 32x32 output tile
    csc485b::a2::gpu::two_hop_reachability<<< grid, block >>>( g );
    CHECK_ERROR( "two_hop_reachability launch" );

    CUDA_CALL( cudaDeviceSynchronize() );
    auto const end = std::chrono::high_resolution_clock::now();

    std::cout << "Build time: "
            << std::chrono::duration_cast<std::chrono::microseconds>(reachability_start - build_start).count()
            << " us"
            << std::endl;

    std::cout << "Reachability time: "
                << std::chrono::duration_cast<std::chrono::microseconds>(end - reachability_start).count()
                << " us"
                << std::endl;
                
}

/**
 * Allocates space for a dense graph and then runs the test code on it.
 */
void run_dense( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m )
{
    using namespace csc485b;

    // allocate device DenseGraph
    a2::node_t * d_matrix;
    cudaMalloc( (void**)&d_matrix, sizeof( a2::node_t ) * n * n );
    a2::DenseGraph d_dg{ n, d_matrix };

    run( d_dg, d_edges, m );

    // check output?
    std::vector< a2::node_t > host_matrix( d_dg.matrix_size() );
    a2::DenseGraph dg{ n, host_matrix.data() };
    cudaMemcpy( dg.adjacencyMatrix, d_dg.adjacencyMatrix, sizeof( a2::node_t ) * d_dg.matrix_size(), cudaMemcpyDeviceToHost );
    std::copy( host_matrix.cbegin(), host_matrix.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );

    // clean up
    cudaFree( d_matrix );
}

/**
 * Allocates space for a sparse graph and then runs the test code on it.
 */
void run_sparse( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m )
{
    using namespace csc485b;

    // allocate device SparseGraph
    a2::node_t * d_offsets, * d_neighbours;
    cudaMalloc( (void**)&d_offsets,    sizeof( a2::node_t ) * n );
    cudaMalloc( (void**)&d_neighbours, sizeof( a2::node_t ) * m );
    a2::SparseGraph d_sg{ n, m, d_offsets, d_neighbours };

    run( d_sg, d_edges, m );

    // copy back and print the CSR
    std::vector< a2::node_t > h_offsets( n ), h_neighbours( m );
    CUDA_CALL( cudaMemcpy( h_offsets.data(),    d_sg.neighbours_start_at, sizeof( a2::node_t ) * n, cudaMemcpyDeviceToHost ) );
    CUDA_CALL( cudaMemcpy( h_neighbours.data(), d_sg.neighbours,          sizeof( a2::node_t ) * m, cudaMemcpyDeviceToHost ) );
    std::cout << "\noffsets: ";
    std::copy( h_offsets.cbegin(), h_offsets.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    std::cout << "\nneighbours: ";
    std::copy( h_neighbours.cbegin(), h_neighbours.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );
    std::cout << std::endl;

    // clean up
    cudaFree( d_neighbours );
    cudaFree( d_offsets );
}

int main()
{
    using namespace csc485b;

    // Create input
    std::size_t constexpr n = 8;
    std::size_t constexpr expected_degree = n >> 1;

    a2::edge_list_t const graph = a2::generate_graph( n, n * expected_degree );
    std::size_t const m = graph.size();

    // lazily echo out input graph
    for( auto const& e : graph )
    {
        std::cout << "(" << e.x << "," << e.y << ") ";
    }

    // allocate and memcpy input to device
    a2::edge_t * d_edges;
    cudaMalloc( (void**)&d_edges, sizeof( a2::edge_t ) * m );
    cudaMemcpyAsync( d_edges, graph.data(), sizeof( a2::edge_t ) * m, cudaMemcpyHostToDevice );

    // run your code!
    run_dense ( d_edges, n, m );
    run_sparse( d_edges, n, m );

    return EXIT_SUCCESS;
}

In [208]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

(0,2) (2,0) (0,3) (3,0) (0,4) (4,0) (0,5) (5,0) (0,6) (6,0) (1,3) (3,1) (1,4) (4,1) (1,5) (5,1) (1,7) (7,1) (2,3) (3,2) (2,5) (5,2) (3,4) (4,3) (3,5) (5,3) (3,6) (6,3) (4,5) (5,4) (4,6) (6,4) (4,7) (7,4) (5,6) (6,5) (5,7) (7,5) (6,7) (7,6) Build time: 2 us
Reachability time: 0 us
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 Build time: 0 us
Reachability time: 0 us

